# 4.4 - GPT & Decoder Models: Writing One Word at a Time

**Module 4 - Model Architecture** | Netsetos GenAI Engineering

This notebook builds a decoder-model's text generator from the ground up: the causal next-word loop that powers every chatbot, the three decoding knobs (temperature, top-k, top-p) coded in pure Python, then the same knobs on a real GPT-2 and a production Gemini API. It closes with reference tables on the GPT family, scaling laws, the 2026 model landscape, logprobs, the training pipeline, and daily production controls.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Minimal environment prep for the notebook. The heavy libraries (`transformers`, `torch`, `google-genai`) are installed only if you uncomment the pip line; this cell just imports NumPy and pins the random seed so the from-scratch sampling cells are reproducible.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic numpy openai torch -q google-genai transformers

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

**Explanation**

A tiny bootstrap cell, not a model call. It imports NumPy and seeds its random generator so that every `np.random.choice` later in the notebook (CLM sampling, top-p sampling) draws the same sequence each run, making the printed outputs stable and gradeable.

**How the code works - step by step**
- **commented `pip install`** - one-line installer for `anthropic`, `numpy`, `openai`, `torch`, `google-genai`, `transformers`; left commented so it only runs in a fresh Colab / environment.
- **`import numpy as np`** - the only library actually imported here; everything else is imported locally in the cell that needs it.
- **`np.random.seed(42)`** - fixes the global NumPy RNG so weighted random picks are reproducible.

**In one line:** import NumPy and lock the seed so the sampling demos are repeatable.

**What you'll see:** (no output - environment prep)

## 1 - Causal Language Modeling (CLM), the one idea behind GPT

BERT guesses hidden words; GPT guesses the NEXT word using only what came before. This cell fakes that with a hand-written probability table instead of a neural net, so you can watch the generate-one-word-at-a-time loop with your own eyes. It's the exact loop GPT-5, Claude and Gemini run - just 7 words instead of 100,000.

In [ ]:
# =============================================
# CLM: Causal Language Modeling
# Given previous words → predict the next word.
# This is how GPT-5, Gemini, Claude all learn.
# =============================================

import numpy as np

# Imagine a tiny vocabulary
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran"]

# A very simple "language model" that just counts patterns
# After seeing "the", what usually comes next?
next_word_probs = {
    "the": {"cat": 0.4, "dog": 0.3, "mat": 0.3},
    "cat": {"sat": 0.6, "ran": 0.4},
    "dog": {"sat": 0.3, "ran": 0.7},
    "sat": {"on": 0.9, "the": 0.1},
    "on":  {"the": 0.95, "mat": 0.05},
}

# Generate text one word at a time!
current = "the"
sentence = [current]

print("Generating text word-by-word:\n")
for _ in range(5):
    if current not in next_word_probs:
        break
    
    options = next_word_probs[current]
    words = list(options.keys())
    probs = list(options.values())
    
    # Show the probabilities
    print(f"  After '{current}', options:")
    for w, p in zip(words, probs):
        bar = "#" * int(p * 20)
        print(f"    {w:6s} {p:.0%} {bar}")
    
    # Pick the next word randomly based on probabilities
    next_word = np.random.choice(words, p=probs)
    sentence.append(next_word)
    current = next_word
    print(f"  → Picked: '{next_word}'\n")

print(f"Generated: {' '.join(sentence)}")
print("\nRun it again - you'll get a different sentence!")
print("That's CLM: predict next word, pick one, repeat.")

# Output:
#   After 'the': cat 40% | dog 30% | mat 30%  -> picked 'cat'
#   Generated: the cat sat on the mat

**Explanation**

A toy stand-in for a language model: a Python dict maps each word to the probabilities of what follows it, and a loop samples the next word, appends it, and repeats. It demonstrates the CLM loop mechanically - no training, no tensors - so the concept is separated from the machinery. Because the pick is random (weighted), rerunning gives a different sentence.

**How the code works - step by step**
- **`vocab` + `next_word_probs`** - a 7-word vocabulary and a hand-coded transition table: after `"the"`, `"cat"` is 40% likely, `"dog"` 30%, `"mat"` 30%, and so on.
- **`current = "the"`, `sentence = [current]`** - seed the loop with a starting word.
- **`for _ in range(5)`** - up to five generation steps; `break`s early if the current word has no known successors (e.g. `"mat"`, `"ran"`).
- **print the options** - draws a text bar chart of each candidate's probability for the current word.
- **`np.random.choice(words, p=probs)`** - the actual sampling step: pick the next word weighted by its probability, append it, make it the new `current`.
- **final prints** - shows the assembled sentence and reminds you a rerun yields a different one.

**In one line:** look up next-word odds -> sample one -> feed it back -> repeat, which is CLM in miniature.

**What you'll see:** For each step it prints the candidate words with percentage bars and the word it picked, then a final assembled sentence such as `Generated: the cat sat on the mat` - and it changes on rerun because the pick is random.

## 2 - The autoregressive loop on a real model

Now swap the toy dict for an actual neural network. This cell loads GPT-2 (124M params, runs on any laptop) and runs the same loop by hand: feed all tokens in, read the next-token probabilities, pick the top one, append, repeat. This is literally how ChatGPT writes - just at greater scale.

> **Downloads the GPT-2 weights** on first run (~500MB); no API key needed, CPU is fine.

In [ ]:
# =============================================
# THE AUTOREGRESSIVE LOOP
# This is EXACTLY how ChatGPT generates text.
# We'll do it step by step so you can see
# each word being chosen.
# =============================================

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 (small version, runs on any laptop)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# Our starting text (the "prompt")
prompt = "Once upon a time, there was a"

# Convert text to token IDs
input_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'\n")
print("Generating one word at a time:\n")

# Generate 20 new words, one at a time
for step in range(20):
    with torch.no_grad():
        # GPT looks at ALL previous tokens
        outputs = model(input_ids)
        
        # Get probabilities for the NEXT token only
        next_token_logits = outputs.logits[0, -1, :]  # last position
        probs = torch.softmax(next_token_logits, dim=-1)
        
        # Pick the most likely token (greedy)
        next_token_id = probs.argmax().unsqueeze(0).unsqueeze(0)
        next_word = tokenizer.decode(next_token_id[0])
        
        # Show what's happening
        confidence = probs.max().item()
        print(f"  Step {step+1:2d}: '{next_word}' ({confidence:.0%} confident)")
        
        # Add this token to the input for the next round
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)

# Show the full generated text
full_text = tokenizer.decode(input_ids[0])
print(f"\nFull output:\n  '{full_text}'")

# Output:
#   Step  1: ' little' (9% confident) ... Step 20: ' day' (14% confident)
#   Full output: 'Once upon a time, there was a little girl who lived ...'

**Explanation**

The real autoregressive loop, unrolled step by step instead of hidden inside `.generate()`. Each iteration runs a full forward pass over the growing sequence, takes the logits at the last position only, softmaxes them, and greedily picks the single most likely token. Doing it manually makes visible that the input gets one token longer every step and that greedy decoding just takes the argmax.

**How the code works - step by step**
- **load `GPT2Tokenizer` + `GPT2LMHeadModel`** - the `"gpt2"` checkpoint; `model.eval()` turns off dropout for deterministic inference.
- **`tokenizer.encode(prompt, return_tensors="pt")`** - turns the prompt string into a tensor of token IDs.
- **`for step in range(20)`** - generate 20 new tokens, one per iteration, under `torch.no_grad()` (no gradients needed at inference).
- **`outputs.logits[0, -1, :]`** - the raw next-token scores at the LAST position only - that's the prediction for the word after everything so far.
- **`torch.softmax(...)` then `.argmax()`** - convert scores to probabilities and greedily take the single most likely token (plus its confidence via `probs.max()`).
- **`torch.cat([input_ids, next_token_id], dim=-1)`** - append the chosen token so the next pass sees a longer context.
- **`tokenizer.decode(input_ids[0])`** - decode the full ID sequence back to readable text.

**In one line:** forward pass -> take last-position logits -> argmax the next token -> append -> repeat 20 times.

**What you'll see:** A numbered trace like `Step 1: ' little' (9% confident)` ... `Step 20: ' day' (14% confident)`, then the full continuation, e.g. `'Once upon a time, there was a little girl who lived ...'`. Greedy decoding makes it deterministic and somewhat repetitive.

## 3 - Temperature, the creativity dial, from scratch

Greedy decoding is safe but dull. Temperature is the fix, and the math is almost embarrassingly simple: divide the logits by a number before softmax. This cell implements that one division and shows how it reshapes the whole next-word distribution across four temperature settings.

In [ ]:
# =============================================
# TEMPERATURE FROM SCRATCH
# It's just ONE line of math. Let's see the
# dramatic effect it has on word choice.
# =============================================

import numpy as np

def softmax(logits):
    """Convert scores to probabilities."""
    e = np.exp(logits - np.max(logits))
    return e / e.sum()

def apply_temperature(logits, temperature):
    """THE ENTIRE TRICK: divide by temperature, then softmax."""
    return softmax(logits / temperature)  # ← That's it. One division.

# GPT's raw scores for the next word after "The cat sat on the"
words  = ["mat", "floor", "roof", "moon", "pizza"]
logits = np.array([5.0,   4.2,    2.5,   0.8,   -1.0])

# Try different temperatures
print("Next word after 'The cat sat on the ___'\n")

for temp in [0.1, 0.5, 1.0, 2.0]:
    probs = apply_temperature(logits, temp)
    
    # What kind of temperature is this?
    if temp <= 0.3:   mood = "Very safe (always picks 'mat')"
    elif temp <= 0.7: mood = "Balanced (usually 'mat', sometimes 'floor')"
    elif temp <= 1.2: mood = "Creative (could pick 'roof' or 'moon')"
    else:             mood = "Wild! (might even pick 'pizza')"
    
    print(f"  Temperature = {temp}")
    print(f"  {mood}")
    for w, p in zip(words, probs):
        bar = "#" * int(p * 40)
        print(f"    {w:8s} {p:5.1%} {bar}")
    print()

# What you'll see:
# temp=0.1 → "mat" gets ~100%, everything else ~0%
# temp=0.5 → "mat" ~83%, "floor" ~17%, others tiny
# temp=1.0 → "mat" ~65%, "floor" ~29%, "roof" ~5%
# temp=2.0 → flatter: "mat" ~47%, "floor" ~32%, "roof" ~14%, even "pizza" ~2%

# Output:
#   temp=0.1 -> mat 100.0%   |   temp=1.0 -> mat 64.6%, floor 29.0%, roof 5.3%
#   temp=2.0 -> flatter: mat 47.0%, floor 31.5%, roof 13.5%, pizza 2.3%

**Explanation**

A pure-NumPy demonstration that temperature is a single division, not a mysterious knob. It defines `softmax` and an `apply_temperature` that divides logits by T first, then applies the same fixed logits at four temperatures to show the effect: low T sharpens the distribution toward the top word, high T flattens it so rare words get a real chance. No model is called - it operates on a hand-picked logit vector so the effect is isolated and clear.

**How the code works - step by step**
- **`softmax(logits)`** - standard numerically-stable softmax (subtract the max before exponentiating).
- **`apply_temperature(logits, temperature)`** - the entire trick: `softmax(logits / temperature)`; one division does all the work.
- **`words` / `logits`** - five candidate next words for "The cat sat on the ___" with fixed raw scores (`mat` 5.0 down to `pizza` -1.0).
- **`for temp in [0.1, 0.5, 1.0, 2.0]`** - sweep four temperatures; a small `mood` label describes each regime.
- **bar-chart print** - shows each word's probability at that temperature so you can watch the top word's dominance shrink as T rises.

**In one line:** divide the logits by T, softmax, and watch low T concentrate probability while high T spreads it.

**What you'll see:** Four blocks of probability bars. At `temp=0.1`, `mat` is ~100%; at `temp=1.0`, `mat` ~65%, `floor` ~29%, `roof` ~5%; at `temp=2.0` it flattens to `mat` ~47%, `floor` ~32%, `roof` ~14%, and even `pizza` ~2%.

## 4 - Top-k and top-p, filtering before you pick

Temperature reshapes the odds but never removes the absurd options. Top-k and top-p do that: they prune the candidate set BEFORE sampling. This cell implements both from scratch on a fixed distribution so you can see exactly which words survive and which get thrown away.

In [ ]:
# =============================================
# TOP-K and TOP-P SAMPLING
# Filter out the bad options, THEN pick randomly
# from the remaining good ones.
# =============================================

import numpy as np

words = ["mat", "floor", "roof", "moon", "pizza", "quantum", "banana"]
probs = np.array([0.35, 0.28, 0.18, 0.10, 0.05, 0.03, 0.01])

print("Original probabilities:")
for w, p in zip(words, probs):
    print(f"  {w:10s} {p:.0%} {'#' * int(p*30)}")

# ── TOP-K: Keep only the best K options ──
def top_k(words, probs, k=3):
    """Keep only the top k highest-probability words."""
    top_indices = np.argsort(probs)[-k:]  # indices of top k
    mask = np.zeros_like(probs)
    mask[top_indices] = probs[top_indices]
    mask /= mask.sum()  # re-normalize
    return mask

print("\nAfter Top-k = 3 (keep best 3, throw away rest):")
tk = top_k(words, probs, k=3)
for w, p in zip(words, tk):
    marker = " <-- kept" if p > 0 else " (removed)"
    print(f"  {w:10s} {p:.0%}{marker}")

# ── TOP-P: Keep words until cumulative prob reaches P ──
def top_p(words, probs, p=0.85):
    """Keep adding words until their total probability reaches p."""
    sorted_idx = np.argsort(probs)[::-1]  # best first
    cumulative = 0
    keep = []
    for idx in sorted_idx:
        keep.append(idx)
        cumulative += probs[idx]
        if cumulative >= p:
            break
    mask = np.zeros_like(probs)
    mask[keep] = probs[keep]
    mask /= mask.sum()
    return mask

print(f"\nAfter Top-p = 0.85 (keep until 85% covered):")
tp = top_p(words, probs, p=0.85)
cumul = 0
for w, p in zip(words, tp):
    if p > 0:
        cumul += p
        print(f"  {w:10s} {p:.0%}  (running total: {cumul:.0%})")
    else:
        print(f"  {w:10s}  -- (removed)")

# ── PICK A WORD using the filtered probabilities ──
print("\nSampling 10 words with Top-p = 0.85:")
for _ in range(10):
    choice = np.random.choice(words, p=tp)
    print(f"  → {choice}")

# Output:
#   Top-k=3 keeps mat/floor/roof; Top-p=0.85 keeps mat/floor/roof/moon
#   Sampling 10x with Top-p: mat, floor, mat, roof, mat, floor, ...

**Explanation**

Two small NumPy filters plus a sampling demo, showing that top-k and top-p are just different ways of trimming the tail before you draw. `top_k` keeps a fixed number of the highest-probability words; `top_p` (nucleus) keeps the smallest set whose cumulative probability crosses a threshold. Both then re-normalize the survivors so they sum to 1, and the cell samples from the top-p set to prove the trimmed words can no longer appear.

**How the code works - step by step**
- **`words` / `probs`** - seven candidates with fixed probabilities from 35% (`mat`) down to 1% (`banana`), printed as a bar chart.
- **`top_k(words, probs, k=3)`** - `np.argsort` to find the top-3 indices, zero out the rest via a mask, then re-normalize the kept three.
- **`top_p(words, probs, p=0.85)`** - sort best-first, accumulate probability adding words until the running total reaches 0.85, keep only those, re-normalize.
- **print of each filter** - marks which words are `kept` vs `removed`, and for top-p shows the running cumulative total as each word is added.
- **`for _ in range(10): np.random.choice(words, p=tp)`** - draw 10 samples from the top-p-filtered distribution to show only surviving words ever get picked.

**In one line:** top-k keeps a fixed count, top-p keeps a cumulative-probability budget, both re-normalize, then you sample from what's left.

**What you'll see:** Top-k=3 keeps `mat`/`floor`/`roof` and marks `moon`/`pizza`/`quantum`/`banana` removed; top-p=0.85 keeps `mat`/`floor`/`roof`/`moon` (running total ~91%); then 10 sampled words, all drawn from those four, e.g. `mat, floor, mat, roof, ...`.

## 5 - Putting it together on real GPT-2

You've built the knobs by hand; now turn them on a real model through one API call. This cell feeds GPT-2 the same prompt five times with five different decoding configs and prints the results side by side, so the abstract settings become visible as different text.

> **Downloads / uses the GPT-2 weights** locally; no API key needed, CPU is fine.

In [ ]:
# =============================================
# GENERATE TEXT with different settings
# Same prompt, different parameters → 
# completely different outputs!
# =============================================

from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

settings = [
    {"name": "Greedy (boring but safe)",
     "args": {"do_sample": False}},
    
    {"name": "Low temperature (factual)",
     "args": {"do_sample": True, "temperature": 0.3, "top_k": 50}},
    
    {"name": "Medium temperature (balanced)",
     "args": {"do_sample": True, "temperature": 0.7, "top_p": 0.9}},
    
    {"name": "High temperature (creative)",
     "args": {"do_sample": True, "temperature": 1.2, "top_p": 0.95}},
    
    {"name": "Very high temperature (wild!)",
     "args": {"do_sample": True, "temperature": 1.8, "top_k": 100}},
]

print(f"Prompt: '{prompt}'\n")
print("=" * 60)

for s in settings:
    output = model.generate(
        input_ids,
        max_new_tokens=40,
        pad_token_id=tokenizer.eos_token_id,
        **s["args"]
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print(f"\n{s['name']}:")
    print(f"  {text}\n")
    print("-" * 60)

print("""
Notice how:
  - Greedy: repetitive, safe, always the same output
  - Low temp: coherent and focused, slight variation
  - Medium: good balance of creativity and coherence
  - High: more creative but sometimes odd word choices
  - Very high: often produces nonsense or bizarre tangents

This is why prompt engineering matters - even the SAME model
gives wildly different results with different settings!
""")

# Output:
#   Greedy:  'The future of artificial intelligence is uncertain. The future ...'
#   Balanced:'The future of artificial intelligence is collaborative, helping ...'

**Explanation**

A comparison harness, not new theory. It reuses GPT-2 but this time calls the high-level `model.generate()` with five preset argument bundles - greedy, low, medium, high, and very-high temperature (mixing in top_k / top_p) - to demonstrate that identical model + identical prompt still produce wildly different text depending on the sampling settings. This is the payoff cell: the knobs from steps 3-4 as real generation arguments.

**How the code works - step by step**
- **load GPT-2 + encode the prompt** - `"The future of artificial intelligence is"` as token IDs.
- **`settings` list** - five named configs: greedy (`do_sample=False`), low temp 0.3 + top_k 50, medium 0.7 + top_p 0.9, high 1.2 + top_p 0.95, very-high 1.8 + top_k 100.
- **`for s in settings`** - each iteration calls `model.generate(..., max_new_tokens=40, **s["args"])` and decodes the result with special tokens stripped.
- **`pad_token_id=tokenizer.eos_token_id`** - silences the pad-token warning for GPT-2, which has no dedicated pad token.
- **closing print** - summarizes the trend: greedy = repetitive, higher temp = more creative but riskier.

**In one line:** same prompt, five `generate()` configs, five distinctly different continuations.

**What you'll see:** Five labeled continuations of the same prompt. Greedy is repetitive (`'...is uncertain. The future ...'`), the balanced setting is more coherent-creative (`'...is collaborative, helping ...'`), and the very-high-temperature one drifts toward nonsense - exact wording varies per run for the sampled configs.

## 6 - The same concepts on a production API (Gemini)

GPT-2 was the teaching model; real apps call hosted APIs. This cell shows that temperature and top_p mean exactly the same thing on Gemini - you're now setting them with understanding, not by copy-paste. It runs one prompt at three temperature/top_p settings.

> **Needs a Gemini API key** (set `GEMINI_API_KEY`). Calls `gemini-3-flash` in the cloud (paid).

In [ ]:
# =============================================
# SAME CONCEPTS WITH GEMINI API
# Now you know what these parameters DO under
# the hood - use them with confidence!
# =============================================

import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

prompt = "Write a one-paragraph story about a robot learning to cook."

configs = [
    ("Safe & factual",     0.2, 0.5),   # (name, temperature, top_p)
    ("Balanced",           0.7, 0.9),
    ("Creative",           1.2, 0.95),
]

for name, temp, top_p in configs:
    response = client.models.generate_content(
        model="gemini-3-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temp,       # You now know: logits / temp
            top_p=top_p,            # You now know: keep until cumulative prob
            max_output_tokens=150,
        ),
    )

    print(f"\n{'='*50}")
    print(f"{name} (temp={temp}, top_p={top_p})")
    print(f"{'='*50}")
    print(response.text)

print("""
You now understand EXACTLY what these parameters do:

  temperature=0.2 → divides scores by 0.2 → top word dominates
  temperature=1.2 → divides scores by 1.2 → more words get a chance
  top_p=0.5 → only considers words covering 50% of probability
  top_p=0.95 → considers words covering 95% of probability

You're no longer blindly copy-pasting parameters.
You KNOW what they do. That's the power of understanding.
""")

# Output:
#   Safe & factual -> a measured, literal paragraph about the robot
#   Creative       -> a vivid, surprising paragraph with twists

**Explanation**

A production-API mirror of steps 3-5. It creates a `genai.Client`, then loops over three (temperature, top_p) presets, passing them through `GenerateContentConfig` on a `gemini-3-flash` call. The point is continuity: the knobs you dissected in pure Python are the identical `temperature` and `top_p` fields every commercial API exposes - only the model size and hosting change.

**How the code works - step by step**
- **`genai.Client(api_key=os.environ["GEMINI_API_KEY"])`** - authenticate from an environment variable, never a hard-coded key.
- **`configs` list** - three presets as `(name, temperature, top_p)`: safe 0.2/0.5, balanced 0.7/0.9, creative 1.2/0.95.
- **`for name, temp, top_p in configs`** - one `client.models.generate_content` call per preset, with `GenerateContentConfig(temperature=..., top_p=..., max_output_tokens=150)`.
- **`print(response.text)`** - shows the generated paragraph under a header naming the settings.
- **closing print** - restates the mechanics you now own: temperature divides the logits, top_p sets the cumulative-probability cutoff.

**In one line:** same temperature/top_p knobs, now as Gemini config fields, across three creativity levels.

**What you'll see:** Three labeled paragraphs answering the robot-cooking prompt: the 0.2/0.5 run reads measured and literal, the 1.2/0.95 run is vivid and surprising. Exact text varies per call since sampling is on. Requires a valid key or the client call errors.

## 7 - The GPT family, from 117M params to multimodal

A reference tour of how GPT evolved and what each generation actually added. This isn't a model call - it's a printed timeline that explains WHY modern APIs behave the way they do, and previews the reasoning shift (o1/o3/GPT-5) that reshaped the field after ChatGPT.

In [ ]:
# =============================================
# THE GPT FAMILY - What changed at each step
# =============================================

gpt_family = [
    ("GPT-1",    2018, "117M",   "Proved: decoder pre-training + fine-tuning works"),
    ("GPT-2",    2019, "1.5B",   "Proved: scale → zero-shot abilities (no fine-tuning!)"),
    ("GPT-3",    2020, "175B",   "Proved: in-context learning (few-shot from prompt)"),
    ("InstructGPT",2022,"~1.3B", "Added RLHF → follows instructions, less harmful"),
    ("ChatGPT",  2022, "~20B?",  "InstructGPT + dialogue fine-tuning → the product"),
    ("GPT-4",    2023, "~1.8T?", "MoE architecture, multimodal (text + images)"),
    ("GPT-4o",   2024, "smaller","Omni: text + images + audio, 2x cheaper, 2x faster"),
    ("o1",       2024, "unknown","Reasoning: a hidden chain-of-thought before answering"),
    ("o3/o4-mini",2025,"unknown","Test-time compute scaling; tools inside the reasoning"),
    ("GPT-5",    2025, "unknown","Unified: routes between fast and thinking modes"),
]

print(f"{'Model':<14} {'Year':>5} {'Params':>8} Key Innovation")
print("-" * 75)
for name, year, params, innovation in gpt_family:
    print(f"  {name:<12} {year:>5} {params:>8} {innovation}")

print("""
THE KEY TRANSITIONS:
  GPT-1 → GPT-2:  Scale unlocks zero-shot abilities (no fine-tuning needed)
  GPT-2 → GPT-3:  In-context learning (put examples IN the prompt)
  GPT-3 → ChatGPT: RLHF alignment (Module 9) makes it follow instructions
  ChatGPT → GPT-4: MoE + multimodal (images, audio) → Module 6
  GPT-4 → o1/o3:   Reasoning - spend compute at inference (think before answering)
  o3 → GPT-5:      Unify - one model routes between fast replies and deep thinking

The training pipeline (preview of Module 9):
  1. Pre-training:   Predict next token on internet text ($$$$)
  2. SFT:            Fine-tune on instruction-response pairs
  3. RLHF/DPO:       Human preference alignment → helpful, harmless, honest
  4. Reasoning:       RL on verified traces (o1, o3, GPT-5, DeepSeek-R1)
""")

# Output:
#   GPT-1  2018   117M  Proved decoder pre-training works
#   GPT-5  2025   unknown  Unified: routes fast vs thinking

**Explanation**

A pure data-and-print reference cell, no model or API involved. A list of tuples records each GPT milestone (year, parameter count, the one insight it proved), and the loop formats it as a table followed by prose on the key transitions and the training pipeline. Read it as a lookup table plus commentary, not runnable ML.

**How the code works - step by step**
- **`gpt_family` list** - one tuple per milestone from GPT-1 (2018, 117M) through GPT-5 (2025), each tagged with its defining innovation (zero-shot, in-context learning, RLHF, MoE, reasoning, unified routing).
- **header + `for` loop** - prints the tuples as an aligned `Model / Year / Params / Key Innovation` table.
- **`THE KEY TRANSITIONS` block** - a multiline string narrating what each jump unlocked (scale -> zero-shot, RLHF -> instruction-following, etc.).
- **training-pipeline preview** - lists the four stages (pre-train -> SFT -> RLHF/DPO -> reasoning) as a forward pointer to Module 9.

**In one line:** a printed timeline of GPT milestones and the single insight each one added.

**What you'll see:** An aligned table of GPT models with years, parameter counts and innovations (`GPT-1 2018 117M ...` down to `GPT-5 2025 unknown ...`), followed by prose on the key transitions and the four-stage training pipeline. No model call - it's a reference table.

## 8 - Scaling laws and emergent abilities

Why do labs keep making models bigger? Because performance follows predictable power laws - and some abilities appear to switch on suddenly at scale. This reference cell contrasts the Kaplan and Chinchilla eras and shows how LLaMA 3 pushed "overtraining" to an extreme for cheap inference.

In [ ]:
# =============================================
# SCALING LAWS - How size → performance
# =============================================

# Kaplan Scaling Laws (2020) - OpenAI
# "Performance follows power law with model size, data, compute"
# 10x more compute → scale model 5.5x, data 1.8x
# This led to GPT-3: 175B params but only 300B tokens (undertrained!)

# Chinchilla Scaling Laws (2022) - DeepMind
# "Data matters more than we thought!"
# Optimal: 20 tokens per parameter
# GPT-3 should have been either 20x smaller OR trained on 20x more data

scaling = [
    ("GPT-3",     175,   300,    1.7,   "Undertrained by Chinchilla standards"),
    ("Chinchilla", 70,   1400,   20.0,  "Optimal: matched GPT-3 quality at 2.5x less cost"),
    ("LLaMA 1",   7,     1000,   142.0, "Overtrained: better at inference (cheaper to serve)"),
    ("LLaMA 3",   8,     15000,  1875.0,"Massively overtrained: best 8B model ever"),
]

print(f"{'Model':<12} {'Params(B)':>10} {'Tokens(B)':>10} {'Tok/Param':>10} Strategy")
print("-" * 75)
for name, params, tokens, ratio, desc in scaling:
    print(f"  {name:<10} {params:>10} {tokens:>10} {ratio:>10.1f} {desc}")

print("""
EMERGENT ABILITIES - capabilities that "switch on" at scale:
  ~13B params: Few-shot arithmetic, word unscrambling
  ~62B params: Chain-of-thought reasoning
  ~175B params: In-context learning, code generation
  
  But recent research (2023-2025) shows: "emergence" may be a
  measurement artifact - smooth improvement looks discontinuous
  when using binary (exact match) metrics. With continuous metrics,
  improvements are gradual. Still, scale clearly helps.

IN-CONTEXT LEARNING - GPT's superpower:
  Prompt: "Translate: cat→gato, dog→perro, house→"
  Output: "casa"
  
  GPT learned translation from 2 examples IN THE PROMPT.
  No weight updates. No fine-tuning. Just pattern recognition
  from pre-training on billions of text patterns.
  This is why prompt engineering (Module 5) is so powerful.
""")

# Output:
#   GPT-3      175    300    1.7  Undertrained by Chinchilla
#   LLaMA 3      8  15000 1875.0  Massively overtrained

**Explanation**

Another print-only reference cell built from a small data table. It encodes four models with their parameter counts, training tokens, and tokens-per-parameter ratio to illustrate the shift from Kaplan (bigger models) to Chinchilla (more data) to LLaMA-style overtraining. The commentary covers emergent abilities and in-context learning conceptually - no computation happens beyond formatting.

**How the code works - step by step**
- **comment block** - summarizes Kaplan (2020) 'scale the model' vs Chinchilla (2022) 'scale the data, ~20 tokens/param'.
- **`scaling` list** - four tuples (GPT-3, Chinchilla, LLaMA 1, LLaMA 3) with params (B), tokens (B), tokens-per-param ratio, and a strategy note.
- **header + `for` loop** - prints the aligned table showing GPT-3 undertrained at 1.7 tok/param vs LLaMA 3 massively overtrained at 1875.
- **`EMERGENT ABILITIES` prose** - lists capabilities that appear at scale and the caveat that 'emergence' may be a metric artifact.
- **`IN-CONTEXT LEARNING` prose** - the cat->gato translation example showing learning from prompt examples with no weight updates, a pointer to Module 5.

**In one line:** a table contrasting model-size vs data-size scaling, plus notes on emergent abilities and in-context learning.

**What you'll see:** A `Model / Params / Tokens / Tok/Param / Strategy` table (`GPT-3 175 300 1.7 ...` up to `LLaMA 3 8 15000 1875.0 ...`), then prose on emergent abilities and an in-context-learning example. No model call - it's a reference table.

## 9 - The 2026 decoder landscape, open vs closed

A practical map of the models you'll actually choose between in production: who makes them, whether the weights are open, rough sizes, and per-million-token prices (verified June 2026). It ends with a decision framework for picking one by task and budget.

In [ ]:
# =============================================
# 2026 DECODER MODEL LANDSCAPE (prices verified June 2026)
# =============================================

models = [
    # (Name, Provider, Type, Size, $/M input, $/M output, Notes)
    ("GPT-5.5",         "OpenAI",    "Closed", "unknown",  5.00,  30.00, "Unified: fast vs thinking router"),
    ("Claude Opus 4.8", "Anthropic", "Closed", "unknown", 5.00,  25.00, "Top-tier code + agentic runs"),
    ("Claude Sonnet 4.6","Anthropic","Closed","unknown", 3.00,  15.00, "Workhorse: quality per dollar"),
    ("Gemini 3 Flash",  "Google",    "Closed", "unknown", 0.50,  3.00,  "Cheap, fast, 1M context, MM"),
    ("Llama 4 Scout",   "Meta",      "Open",   "17B/16E", 0.00,  0.00,  "Self-host free; 10M context"),
    ("DeepSeek R1/V4", "DeepSeek",  "Open",   "671B/37B",0.55, 2.19,  "Open-weights reasoning, MoE"),
    ("Qwen 3",         "Alibaba",   "Open",   "235B/22B",0.00, 0.00,  "Apache 2.0, multilingual"),
    ("Gemma 3 27B",    "Google",    "Open",   "27B",     0.00,  0.00,  "Best laptop-class open model"),
]

print(f"{'Model':<16} {'Provider':<11} {'Type':<7} {'Size':<10} {'$/M In':>7} {'$/M Out':>8}")
print("-" * 70)
for name, prov, typ, size, inp, out, _ in models:
    print(f"  {name:<14} {prov:<11} {typ:<7} {size:<10} ${inp:>6.2f} ${out:>7.2f}")

print("""
DECISION FRAMEWORK (Module 13 goes deeper):
  Budget app:     Gemini 3 Flash ($0.50/M) - cheap, fast, 1M context
  Best quality:   GPT-5.5 or Claude Opus 4.8 for hard code + reasoning
  Reasoning:      o3 / GPT-5 thinking / DeepSeek R1 - compute at inference
  Self-hosted:    Llama 4 or Qwen 3 (free weights, your GPUs)
  Edge/laptop:    Gemma 3 27B or a quantized Llama 4 Scout
  Indian langs:   Gemini (strong Hindi), Qwen (broad multilingual)
  Fine-tuning:    Llama 4 or Qwen 3 (Module 9)

  Open weights = free to download, run, and fine-tune (you pay compute).
  Closed APIs  = per-token cost, zero infra.
  MoE is everywhere: DeepSeek is 671B total but only 37B active per token.
""")

# Output:
#   GPT-5.5         OpenAI    Closed  $ 5.00  $30.00
#   Gemini 3 Flash  Google    Closed  $ 0.50  $ 3.00

**Explanation**

A print-only reference table of the current model market, not an API call. A list of tuples holds each model's provider, open/closed status, size, and input/output pricing; the loop formats it, and a closing string gives a task-to-model decision guide. Prices and names are a June-2026 snapshot meant for orientation, not a live lookup.

**How the code works - step by step**
- **`models` list** - eight tuples spanning closed APIs (GPT-5.5, Claude Opus 4.8, Claude Sonnet 4.6, Gemini 3 Flash) and open weights (Llama 4 Scout, DeepSeek R1/V4, Qwen 3, Gemma 3 27B), each with provider, type, size, and $/M input and output.
- **header + `for` loop** - prints an aligned table; open models show `$0.00` because you self-host and pay only compute.
- **`DECISION FRAMEWORK` prose** - maps needs to picks (budget -> Gemini Flash, best quality -> GPT-5.5/Claude Opus, reasoning -> o3/R1, self-host -> Llama/Qwen, edge -> Gemma), with a note that MoE (e.g. DeepSeek 671B total / 37B active) is now standard.

**In one line:** a priced open-vs-closed model table plus a task-based picking guide.

**What you'll see:** A `Model / Provider / Type / Size / $/M In / $/M Out` table (e.g. `GPT-5.5 OpenAI Closed $5.00 $30.00`, `Gemini 3 Flash Google Closed $0.50 $3.00`, open models at `$0.00`), followed by the decision-framework prose. No model call - it's a reference table.

## 10 - Logits, logprobs, and seeing what GPT is thinking

The most useful debugging tool you get for free: the probability the model assigned to every possible next token. This cell runs one GPT-2 forward pass and prints the top-5 candidates with their probabilities and log-probabilities, then explains what logprobs buy you in production.

> **Uses the GPT-2 weights** locally; no API key needed.

In [ ]:
# =============================================
# LOGITS → LOGPROBS - See inside the model's head
# =============================================
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

prompt = "The capital of India is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]  # Last position's raw scores

# Logits → probabilities → logprobs
probs = F.softmax(logits, dim=-1)
logprobs = torch.log(probs)

# Top 5 most likely next tokens
top5 = torch.topk(probs, 5)
print("What GPT thinks comes after 'The capital of India is':\n")
for i in range(5):
    token = tokenizer.decode(top5.indices[i])
    prob = top5.values[i].item()
    lp = logprobs[top5.indices[i]].item()
    print(f"  {i+1}. '{token}' → prob={prob:.1%}, logprob={lp:.2f}")

print("""
LOGITS vs LOGPROBS vs PROBABILITIES:
  logits  = raw scores from model (unbounded, e.g. -2.1, 5.3, 1.7)
  probs   = softmax(logits) → sums to 1.0 (e.g. 0.45, 0.30, 0.15)
  logprobs = log(probs) → always negative (e.g. -0.80, -1.20, -1.90)
  
  logprob = 0.0  → 100% confident (never happens in practice)
  logprob = -0.7 → ~50% confident
  logprob = -2.3 → ~10% confident
  logprob = -4.6 → ~1% confident

PRODUCTION USES of logprobs:
  1. Hallucination detection → low logprob on factual claims = flag for review
  2. Classification confidence → use logprobs instead of text parsing
  3. Perplexity = exp(-avg logprob) → standard metric for model quality
     Lower perplexity = model is more certain = better model
  4. Token cost optimization → route low-confidence queries to larger models
""")

# Output:
#   1. ' New'   -> prob=64.2%, logprob=-0.44
#   2. ' Delhi' -> prob=18.1%, logprob=-1.71

**Explanation**

A single-forward-pass inspection cell, not a generation loop. It runs GPT-2 once on a prompt, takes the last-position logits, converts them to probabilities (softmax) and log-probabilities (log), and reports the five most likely next tokens. The point is to make the logits -> probs -> logprobs pipeline concrete - the same pipeline your temperature and top-p code manipulates - and to connect logprobs to real uses like perplexity and hallucination detection.

**How the code works - step by step**
- **load GPT-2 + tokenize** - `"The capital of India is"` into model inputs.
- **`outputs.logits[0, -1, :]`** - grab the raw next-token scores at the last position under `torch.no_grad()`.
- **`F.softmax` then `torch.log`** - turn logits into probabilities, then into logprobs (always negative).
- **`torch.topk(probs, 5)`** - select the five highest-probability tokens; the loop decodes each and prints its probability and logprob.
- **`LOGITS vs LOGPROBS` prose** - a cheat sheet mapping logprob values to confidence (0.0 = 100%, -0.7 ~ 50%, -4.6 ~ 1%).
- **`PRODUCTION USES` prose** - lists hallucination detection, classification confidence, perplexity = exp(-avg logprob), and confidence-based routing.

**In one line:** one forward pass -> softmax -> log -> read off the top-5 next-token probabilities and their logprobs.

**What you'll see:** The top-5 next tokens after the prompt with probability and logprob, e.g. `1. ' New' -> prob=64.2%, logprob=-0.44`, `2. ' Delhi' -> prob=18.1%, logprob=-1.71`, followed by the logits-vs-logprobs cheat sheet and production-use notes.

## 11 - From raw model to ChatGPT, the training pipeline

The story that explains why a 1.3B InstructGPT was preferred over the 175B GPT-3: alignment beats scale. This reference cell lays out the four training stages - pre-training, SFT, RLHF/DPO, reasoning - with what each one costs, consumes, and produces.

In [ ]:
# =============================================
# THE LLM TRAINING PIPELINE - 4 stages
# =============================================

pipeline = {
    "1. Pre-Training": {
        "task": "Next-token prediction on trillions of tokens",
        "data": "Internet text (FineWeb, Common Crawl, books, code)",
        "cost": "$10K-$100M+ (most expensive stage)",
        "output": "Base model - completes text but doesn't follow instructions",
        "example": "'What is 2+2?' → 'What is 2+3? What is 2+4?...' (just continues)",
    },
    "2. SFT (Supervised Fine-Tuning)": {
        "task": "Train on (instruction, ideal_response) pairs",
        "data": "~100K human-written instruction-response examples",
        "cost": "$100-$10K",
        "output": "Instruction-following model - answers questions properly",
        "example": "'What is 2+2?' → '4' (follows instruction format)",
    },
    "3. RLHF / DPO (Alignment)": {
        "task": "Optimize for human preferences",
        "data": "Human rankings: 'Response A is better than B'",
        "cost": "$10K-$1M (human annotators are expensive)",
        "output": "Aligned model - helpful, harmless, honest",
        "methods": "PPO → DPO → GRPO (each simpler than the last)",
    },
    "4. Reasoning (2024+)": {
        "task": "Chain-of-thought fine-tuning + test-time compute",
        "data": "Reasoning traces (verified math, code, logic)",
        "cost": "a large, growing share of the compute budget",
        "output": "Reasoning model - thinks before answering (o1, o3, R1)",
        "key": "DeepSeek R1: GRPO only, no SFT needed for reasoning!",
    },
}

for stage, info in pipeline.items():
    print(f"\n{stage}")
    for k, v in info.items():
        print(f"  {k}: {v}")

print("""
THE INSIGHT: Alignment beats scale.
  InstructGPT (1.3B params + RLHF) was PREFERRED over GPT-3 (175B).
  A small, well-aligned model > a huge, unaligned model.

ALIGNMENT EVOLUTION (each removes a requirement):
  RLHF (2022): Reward model + PPO → 4 models needed
  DPO  (2023): Direct preference → 2 models (no reward model)
  GRPO (2024): Group relative policy → no critic model, lower memory
  ORPO (2024): Single unified objective → 1 model
  
Module 9 covers SFT + RLHF/DPO in depth with hands-on labs.
""")

# Output:
#   1. Pre-Training: next-token prediction on trillions of tokens ...
#   (prints all 4 stages, then 'Alignment beats scale')

**Explanation**

A structured reference cell, no training actually runs. A nested dictionary describes each of the four pipeline stages (task, data, cost, output, example), and the loop pretty-prints it, followed by prose on why alignment beats scale and how alignment methods (PPO -> DPO -> GRPO -> ORPO) each shed a requirement. It's the conceptual backbone that Module 9 will implement.

**How the code works - step by step**
- **`pipeline` dict** - four keys, one per stage, each mapping to a sub-dict of `task` / `data` / `cost` / `output` and an `example` or `methods`/`key` note.
- **stage 1 pre-training** - next-token prediction on trillions of tokens; produces a base model that only continues text.
- **stage 2 SFT** - trains on (instruction, response) pairs so the model answers rather than continues.
- **stage 3 RLHF/DPO** - optimizes on human preference rankings for helpful/harmless/honest behavior.
- **stage 4 reasoning** - chain-of-thought + test-time compute; notes DeepSeek R1 reached reasoning via GRPO with no SFT.
- **nested `for` loops** - print each stage and its fields.
- **closing prose** - 'alignment beats scale' (1.3B InstructGPT > 175B GPT-3) and the alignment-method evolution.

**In one line:** a printed four-stage map - pre-train -> SFT -> RLHF/DPO -> reasoning - showing alignment can beat raw scale.

**What you'll see:** Each of the four stages printed with its task/data/cost/output fields, then prose stating alignment beats scale and listing the RLHF -> DPO -> GRPO -> ORPO progression. No training happens - it's a reference table.

## 12 - Production controls, the features you use daily

The three things a GenAI engineer reaches for constantly: chat templates, structured (JSON-guaranteed) outputs, and streaming. This cell shows all three on Gemini and explains that each is just the CLM loop from step 1 with a wrapper around it.

> **Needs a Gemini API key** (set `GEMINI_API_KEY`). Calls `gemini-3-flash` in the cloud (paid).

In [ ]:
# =============================================
# PRODUCTION CONTROLS - What engineers use daily
# =============================================
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# 1. CHAT TEMPLATES - system/user/assistant format
# Under the hood, this is just concatenated text for next-token prediction:
#   <system>You are a helpful assistant.</system>
#   <user>What is 2+2?</user>
#   <assistant>4</assistant>
# The model does CLM on this ENTIRE sequence - same as Step 1!
# The system instruction rides in the config below.

# 2. STRUCTURED OUTPUTS - force valid JSON
# Internally: constrained decoding masks non-conforming tokens at each step
# If the schema says "age" must be an integer, tokens like "twenty" get
# probability 0.0 - the model CANNOT produce invalid output.

response = client.models.generate_content(
    model="gemini-3-flash",
    contents="Extract: 'Sanjay Kumar, age 35, lives in Hyderabad'",
    config=types.GenerateContentConfig(
        system_instruction="You are a concise technical assistant. Reply in JSON.",
        response_mime_type="application/json",
        response_schema={
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "integer"},
                "city": {"type": "string"},
            }
        },
    ),
)
print(f"Structured output: {response.text}")
# → {"name": "Sanjay Kumar", "age": 35, "city": "Hyderabad"}

# 3. STREAMING - token-by-token delivery (SSE)
# Why streaming? Lower time-to-first-token; studies link it to longer sessions.
# Each token is sent as it's generated via Server-Sent Events.
for chunk in client.models.generate_content_stream(
    model="gemini-3-flash",
    contents="Explain RAG in 3 bullets",
):
    print(chunk.text, end="", flush=True)  # tokens appear one by one

print("""

OTHER PRODUCTION PARAMETERS:
  repetition_penalty: 1.2  → penalizes tokens already generated (prevents loops)
  frequency_penalty:  0.5  → reduces probability of frequent tokens (OpenAI-specific)
  presence_penalty:   0.5  → encourages topic diversity (OpenAI-specific)
  stop_sequences: ["\\n"]  → stop generating when this token appears
  max_tokens: 500          → hard limit on output length
  
DECODING STRATEGIES (beyond temp/top-k/top-p):
  Greedy:       do_sample=False → always pick highest prob (deterministic)
  Beam search:  num_beams=4 → explore multiple paths (best for translation)
  Contrastive:  penalty_alpha → balances quality and diversity
  Min-p:        min_p=0.1 → dynamic threshold (popular in local LLM community)
""")

**Explanation**

A hands-on production-features cell that makes two live Gemini calls - one for structured output, one for streaming - framed with comments that tie each feature back to next-token prediction. Structured outputs use constrained decoding (invalid tokens masked to probability 0), streaming delivers tokens as they're generated over Server-Sent Events, and chat templates are just concatenated system/user/assistant text the model runs CLM over. It closes with a reference list of other decoding parameters and strategies.

**How the code works - step by step**
- **chat-template comment** - explains the system/user/assistant format is concatenated text the model does CLM on; the system instruction rides in the config.
- **structured-output call** - `generate_content` with `response_mime_type="application/json"` and a `response_schema` (name/age/city types) so the model is forced to emit schema-valid JSON via constrained decoding.
- **`print(response.text)`** - shows the guaranteed-valid JSON extraction.
- **streaming call** - `generate_content_stream(...)` looped with `print(chunk.text, end="", flush=True)` so tokens appear one at a time.
- **closing prose** - reference list of `repetition_penalty` / `frequency_penalty` / `presence_penalty` / `stop_sequences` / `max_tokens`, and decoding strategies (greedy, beam search, contrastive, min-p), with a pointer to Module 10's reliance on structured outputs.

**In one line:** JSON-schema-constrained output plus token-by-token streaming on Gemini, both just wrappers around the CLM loop.

**What you'll see:** A structured JSON line like `{"name": "Sanjay Kumar", "age": 35, "city": "Hyderabad"}`, then a 3-bullet RAG explanation printing token-by-token, followed by the reference list of production parameters and decoding strategies. Requires a valid key or the calls error.

You built text generation from its single idea outward: causal next-word prediction, the autoregressive loop that feeds each word back in, and the three decoding knobs - temperature, top-k, top-p - written in plain Python, then verified on GPT-2 and a Gemini API that expose the exact same controls. The reference cells then situate that mechanism in the wider decoder world: the GPT lineage, scaling laws, the 2026 open-vs-closed landscape, logprobs for debugging, the four-stage training pipeline, and the production controls you'll touch daily. Next up in Module 4, attention and positional encoding explain HOW the model computes those next-word scores in the first place; Modules 5, 9, 10, and 13 build directly on the prompt-engineering, fine-tuning, tool-use, and cost themes previewed here.